# Integration cell line - patient with celligner - Procan and FDL data
Use the celligner package to integrate the proteomic data. Uses the Procan_cell_line and FDL_patient (label-free DIA) data for this.

Use the celligner_env environment due to Python needs to be at version 3.9 (the cellLine_patient environment uses Python 3.11). See README.md for instructions on how to set this up.

This is of course the adapted workflow for Celligner on proteomics, as the method was originally developed for transcriptomics data. Here, I used the merged proteomics data, where missing values were replaced by 0. Additionally, I only kept tissues for which at least 20 samples are present in each dataset. Filename: "1_1_9_ProCan_FDL_PanCancer_merged_tissue20.h5ad"

In [1]:
%load_ext autoreload
%autoreload 2
import random
random.seed(42)

import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scanpy as sc
from celligner import Celligner

/share/conda-envs/celligner_env/lib/python3.9/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/share/conda-envs/celligner_env/lib/python3.9/site-packages/mnnpy/utils.py:14: NumbaWarning: 
Compilation is falling back to object mode WITH looplifting enabled because Function "l2_norm" failed type inference due to: No implementation of function Function(<function norm at 0x7281901c8b80>) found for signature:
 
 >>> norm(x=array(float32, 2d, A), axis=Literal[int](1))
 
There are 2 candidate implementations:
  - Of which 2 did not match due to:
  Overload in function 'norm_impl': File: numba/np/linalg.py: Line 2351.
    With argument(s): '(x=array(float32, 2d, A), axis=int64)':
   Rejected as the i

In [ ]:
import yaml
with open("../../config/config.local.yaml", "r") as f:
    config = yaml.safe_load(f)
basedir = config['code_dir']

anndata_input_dir = os.path.join(config['output_data_dir'], 
                         "processed_proteomics_data", 
                         "1_1_9_ProCan_FDL_PanCancer_merged_tissue20.h5ad"
)  # where the tumor and cell line data, in anndata format, are stored

data_output_dir = os.path.join(config['output_data_dir'], "celligner_integration")  # where the output of the protint model will be stored
if not os.path.exists(data_output_dir):
    os.makedirs(data_output_dir)

In [3]:
sys.path.append(os.path.join(basedir, '03_analysisScripts'))
from Utils.color_palette import palette_data_source_label_free as palette_ds

# Celligner integration on the full data
Log2-transformed data, with missing values replaced by 0

In [ ]:
adata = sc.read_h5ad(anndata_input_dir)
adata_cellline = adata[adata.obs['data_source']=='ProCan_cell_line']
adata_patient = adata[adata.obs['data_source']=='FDL_patient']

In [5]:
my_celligner = Celligner()
my_celligner.fit(adata_cellline.to_df())

/share/conda-envs/celligner_env/lib/python3.9/site-packages/anndata/_core/anndata.py:402: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Doing PCA..
Computing neighbors..
Clustering..
Running differential expression on 31 clusters..
Running limmapy..


/share/conda-envs/celligner_env/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:55: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for name, values in obj.iteritems():


In [6]:
my_celligner.transform(adata_patient.to_df())

/share/conda-envs/celligner_env/lib/python3.9/site-packages/anndata/_core/anndata.py:402: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Doing PCA..
Computing neighbors..
Clustering..
Running differential expression on 31 clusters..
Running limmapy..


/share/conda-envs/celligner_env/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:55: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for name, values in obj.iteritems():


Running cPCA..
Regressing top cPCs out of reference dataset..
Regressing top cPCs out of target dataset..
Doing the MNN analysis using Marioni et al. method..
  Looking for MNNs...
  Found 1445 mutual nearest neighbors.
Done


In [7]:
my_celligner.computeMetricsForOutput()

Computing UMAP embedding...


/share/conda-envs/celligner_env/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Computing clusters..
Doing PCA..


/share/conda-envs/celligner_env/lib/python3.9/site-packages/anndata/_core/anndata.py:402: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Computing neighbors..
Clustering..
Computing tumor-CL distance..


In [8]:
for name, val in vars(my_celligner).items():
    if isinstance(val, pd.DataFrame):
        print(name, val.shape, list(val.columns)[:10])

ref_input (771, 6251) ['P31946', 'P62258', 'Q04917', 'P61981', 'P31947', 'P27348', 'P63104', 'Q14738', 'Q16537', 'Q13362']
ref_de_genes (6251, 4) ['AveExpr', 'F', 'P.Value', 'adj.P.Val']
target_input (550, 6251) ['P31946', 'P62258', 'Q04917', 'P61981', 'P31947', 'P27348', 'P63104', 'Q14738', 'Q16537', 'Q13362']
target_de_genes (6251, 4) ['AveExpr', 'F', 'P.Value', 'adj.P.Val']
combined_output (1321, 6251) ['P31946', 'P62258', 'Q04917', 'P61981', 'P31947', 'P27348', 'P63104', 'Q14738', 'Q16537', 'Q13362']
umap_reduced (1321, 2) ['umap1', 'umap2']
tumor_CL_dist (550, 771) ['BC-1', 'L-363', 'EoL-1-cell', 'NCI-H727', 'MV-4-11', 'HSC-39', 'RPMI-8402', 'MDA-MB-453', 'NKM-1', 'KURAMOCHI']


In [ ]:
umap_coord = my_celligner.umap_reduced

# Create mapping: sample → data_source
data_source_map = {}

for s in umap_coord.index:
    if s in adata_cellline.obs_names:
        data_source_map[s] = "ProCan_cell_line"
    elif s in adata_patient.obs_names:
        data_source_map[s] = "FDL_patient"
    else:
        data_source_map[s] = "unknown"

# Add the column
umap_coord["data_source"] = umap_coord.index.map(data_source_map)

# Build a dictionary of sample → organ
organ_map = {}

# cell line data
for s, org in adata_cellline.obs["Tissue type"].items():
    organ_map[s] = org

# tumor data
for s, org in adata_patient.obs["Tissue type"].items():
    organ_map[s] = org

# Map organ into umap_coord
umap_coord["Tissue type"] = umap_coord.index.map(organ_map)

umap_coord


,umap1,umap2,data_source,Tissue type
cell_id,,,,
00FZN,5.072474,9.677983,FDL_patient,brain
00FZL,5.618968,10.437027,FDL_patient,brain
00FZJ,4.816957,10.833796,FDL_patient,brain
00FZ1,5.144199,10.112303,FDL_patient,brain
00FYV,6.131511,9.631137,FDL_patient,brain
...,...,...,...,...
VMRC-RCW,5.479862,3.481306,ProCan_cell_line,kidney
NCI-H1304,0.261985,10.380279,ProCan_cell_line,lung
SC-1,10.375766,5.235391,ProCan_cell_line,haematopoietic and lymphoid


In [12]:
my_celligner.combined_output

UniprotID,P31946,P62258,Q04917,P61981,P31947,P27348,P63104,Q14738,Q16537,Q13362,...,Q6PML9,Q07157,Q9UDY2,O95049,Q9UK55,O75312,O95218,O43264,Q9H900,Q15942
cell_id,,,,,,,,,,,,,,,,,,,,,
00FZN,-0.350946,0.627620,-0.070141,0.379906,-2.115300,1.130492,-0.143000,-0.045377,0.958416,0.076632,...,1.358781,0.461200,-1.260914,-0.795047,0.331401,0.078185,0.136115,-0.028145,-0.149330,0.496751
00FZL,-0.093701,0.033295,0.072550,0.196012,-3.471105,-0.375274,0.076770,0.120586,1.211996,-0.066471,...,0.352497,0.045149,-2.067150,-0.134895,-0.287540,-0.079295,0.199126,0.140827,1.346159,-0.113346
00FZJ,-0.646807,-0.869557,0.054404,-0.548712,-2.391905,-0.605897,-0.339470,0.545489,0.723563,0.816537,...,1.539460,0.216978,-2.279258,-0.510334,2.594313,-1.233816,-3.119730,0.703902,1.284653,0.247050
00FZ1,-0.752984,0.454485,-0.066746,0.291216,-0.821626,1.041325,0.172141,0.608903,-0.021826,-0.940768,...,1.655160,-0.263964,-1.265315,-0.705847,1.065705,-0.179655,-0.504192,0.764277,0.502475,-1.064893
00FYV,-0.234624,0.461466,0.837346,0.316844,-3.578065,0.362925,-0.300200,-0.317002,1.524489,2.734818,...,2.137821,0.094702,-2.155226,-0.555229,0.390683,0.338189,0.272944,0.610461,0.726249,-0.654914
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VMRC-RCW,-0.285504,0.022780,0.247578,0.812192,2.933465,-0.219009,0.323039,-0.067091,-1.638187,-0.852499,...,1.169226,0.395382,1.091605,-0.639743,0.038572,0.252365,0.389067,0.071892,-0.598953,2.639189
NCI-H1304,-0.807236,-0.526305,-0.075685,0.113067,3.314708,-0.571060,-0.871799,0.814868,1.416266,1.555637,...,1.755304,-3.838378,-1.979599,-0.831132,3.246449,-0.264530,0.553110,0.177488,1.505875,-2.374759
SC-1,-0.372038,0.042391,-0.133848,0.368431,-6.686033,-0.336366,-0.769473,0.002471,0.263149,1.637178,...,-0.439164,1.684116,-1.307241,-0.752021,-0.199321,-0.598081,0.363854,0.496686,1.719379,-4.535037


In [13]:
my_celligner.combined_output.to_csv(os.path.join(data_output_dir, "transformed_data_celligner_FDL_Procan.csv"))